# Early Allergy Symptoms With Built healthcare MCP

This notebook continues the healthcare MCP examples with a more complete, stateful story focused on **early allergy symptoms**.

**Story:** Emma has a known grass-pollen allergy. Shortly after a park walk, she develops sneezing, itchy/watery eyes, and a runny nose. A clinic-assistant Agent uses our custom MCP server to reconstruct the timeline, compare allergy-like and flu-like clues, check for emergency warning signs, and create a clinician handoff. We then run an optional escalation scenario to show how the same MCP server responds when a severe symptom is added.

All patient records and rules in this notebook are fictional and educational. This is not a diagnostic or triage system.


## 1. Imports

We use the OpenAI Agents SDK as the MCP client and `MCPServerStdio` to start our local healthcare server.


In [1]:
from dotenv import load_dotenv
from agents import Agent, Runner, trace
from agents.mcp import MCPServerStdio
from IPython.display import display, Markdown

load_dotenv(override=True)


True

## 2. Windows Jupyter MCP helper

Native Windows Jupyter needs a subprocess-capable event loop. We also redirect MCP subprocess stderr to a normal file so Jupyter's virtual stderr does not cause `fileno` errors. Run this setup cell once after every kernel restart.


In [2]:
import asyncio
from contextlib import asynccontextmanager
import agents.mcp.server as agents_mcp_server

try:
    from mcp import stdio_client
except ImportError:
    from mcp.client.stdio import stdio_client

@asynccontextmanager
async def notebook_stdio(params):
    with open("mcp_stderr.log", "a", encoding="utf-8") as log:
        async with stdio_client(params, errlog=log) as streams:
            yield streams

agents_mcp_server.stdio_client = notebook_stdio

def _run_on_proactor(async_fn):
    from asyncio.windows_events import ProactorEventLoop
    loop = ProactorEventLoop()
    asyncio.set_event_loop(loop)
    try:
        result = loop.run_until_complete(async_fn())
        loop.run_until_complete(asyncio.sleep(0.3))
        return result
    finally:
        asyncio.set_event_loop(None)
        loop.close()

async def run_mcp(async_fn):
    return await asyncio.to_thread(_run_on_proactor, async_fn)


## 3. Start the allergy MCP server

Keep `healthcare_allergy_MCP_server.py` in the same folder as this notebook. The server contains fictional patient data, episode state, rule-based comparison logic, a medication-allergy demo check, MCP resources, and a reusable MCP prompt.


In [3]:
params = {
    "command": "uv",
    "args": ["run", "healthcare_allergy_MCP_server.py"]
}

async def list_tools_workflow():
    async with MCPServerStdio(
        params=params,
        client_session_timeout_seconds=60
    ) as server:
        return await server.list_tools()

mcp_tools = await run_mcp(list_tools_workflow)

for tool in mcp_tools:
    print(tool.name)


get_patient_profile
start_allergy_episode
record_episode_symptom
get_episode_timeline
get_allergy_reference
compare_allergy_vs_flu
assess_emergency_warning
check_medication_allergy
create_clinician_handoff


The server exposes a complete allergy-episode workflow:

- `get_patient_profile` — read Emma's allergy-focused profile
- `start_allergy_episode` — create a new episode
- `record_episode_symptom` — add a symptom with severity and onset time
- `get_episode_timeline` — reconstruct the episode chronologically
- `get_allergy_reference` — return concise source-linked medical guidance
- `compare_allergy_vs_flu` — compare simple allergy-like and flu-like clues
- `assess_emergency_warning` — check configured severe-reaction warning signs
- `check_medication_allergy` — demonstrate a medication/allergy rule
- `create_clinician_handoff` — generate a structured visit handoff

The server also exposes two MCP resources and one reusable MCP prompt.


## 4. An early allergy pattern

Emma's fictional episode already contains three early symptoms after grass-pollen exposure. We ask the Agent to add one new mild symptom, inspect the episode, compare allergy versus flu clues, check for emergency warning signs, and prepare a clinician handoff.


In [4]:
instructions = """
You are a clinic information assistant working only with fictional demo records.

Use the MCP tools rather than guessing.
For allergy questions, use get_allergy_reference when useful.
Separate early/common allergy clues from severe allergic-reaction warning signs.
The allergy-versus-flu comparison is only a pattern comparison, not a diagnosis.
If assess_emergency_warning reports an emergency warning, state the emergency action clearly.
Do not invent symptoms, diagnoses, medications, or test results.
Finish with a concise clinician-style summary.
"""

request = """
Emma is preparing for a clinic visit after grass-pollen exposure during a park walk.

Please:
1. Review Emma's profile and episode-001 timeline.
2. Record a new symptom: itchy throat, body system respiratory, severity 3, onset 15 minutes.
3. Review the early-allergy reference.
4. Compare the episode with allergy-like versus flu-like clues.
5. Assess emergency warning signs.
6. Check whether amoxicillin conflicts with Emma's recorded allergies.
7. Create a clinician handoff.
8. Give me a short final summary. Do not diagnose.
"""

model = "gpt-4.1-mini"

async def early_allergy_workflow():
    async with MCPServerStdio(
        params=params,
        client_session_timeout_seconds=60
    ) as mcp_server:
        agent = Agent(
            name="early_allergy_clinic_assistant",
            instructions=instructions,
            model=model,
            mcp_servers=[mcp_server]
        )

        with trace("early_allergy_story"):
            result = await Runner.run(
                agent,
                request,
                max_turns=20
            )

        return result.final_output

final_output = await run_mcp(early_allergy_workflow)
display(Markdown(final_output))


1. Emma is a 42-year-old patient with known allergies to grass pollen and penicillin. Her current medication includes cetirizine. Recent vitals are stable.
2. Her episode-001 started after grass pollen exposure during a park walk. Symptoms so far include sneezing, itchy watery eyes, runny nose, and the newly added itchy throat, all mild in severity (3-4/10) and appearing within 15 minutes of exposure.
3. Early allergic rhinitis symptoms typically include sneezing, runny or stuffy nose, itching in the nose/eyes/throat, and watery red eyes.
4. The symptom pattern is more allergy-like (itching, sneezing, watery eyes, runny nose, onset soon after exposure) rather than flu-like.
5. There are no severe allergic reaction warning signs present currently. Continued monitoring and appropriate follow-up are advised.
6. There is a possible allergy conflict with amoxicillin due to Emma's known penicillin allergy (note this is a demo mapping).
7. A detailed clinician handoff has been created, summarizing the patient's allergy episode and clinical data.

Summary: Emma presents with mild, allergy-consistent symptoms shortly after grass pollen exposure without emergency warning signs. She has a known penicillin allergy which may contraindicate amoxicillin use. Clinical evaluation and monitoring are recommended.

## 5. Follow-up 

The first workflow demonstrates early, relatively mild symptoms. This second fictional scenario shows why symptom **evolution over time** matters: we add shortness of breath later in the same episode and ask the Agent to reassess the emergency-warning logic and regenerate the handoff.


In [5]:
escalation_request = """
This is a fictional training scenario.

Emma now reports shortness of breath, body system respiratory,
severity 8, onset 25 minutes after the same exposure.

1. Record the new symptom in episode-001.
2. Re-check the episode timeline.
3. Reassess emergency warning signs.
4. Create an updated clinician handoff.
5. Return a short response that clearly follows the emergency guidance from the tool.
"""

async def escalation_workflow():
    async with MCPServerStdio(
        params=params,
        client_session_timeout_seconds=60
    ) as mcp_server:
        agent = Agent(
            name="allergy_escalation_assistant",
            instructions=instructions,
            model=model,
            mcp_servers=[mcp_server]
        )

        with trace("allergy_escalation_story"):
            result = await Runner.run(
                agent,
                escalation_request,
                max_turns=12
            )

        return result.final_output

escalation_output = await run_mcp(escalation_workflow)
display(Markdown(escalation_output))


Emma's episode now includes shortness of breath of severity 8, occurring 25 minutes after grass pollen exposure. This is a severe allergic-reaction warning sign. Immediate action is required: use Emma's prescribed emergency plan, including epinephrine if available, and call emergency services without delay.

Summary for clinician:
- Patient: Emma, 42 yo, known grass pollen allergy.
- Trigger: grass pollen exposure during a park walk.
- Symptoms timeline: sneezing (3/10) at 5 min, itchy watery eyes (4/10) at 7 min, runny nose (3/10) at 10 min, shortness of breath (8/10) at 25 min.
- Recent vitals stable.
- Emergency action: Follow emergency anaphylaxis protocol immediately.

## 6. Summmary of this allergy MCP.

This version combines several MCP ideas in one consistent story:

1. **Stateful episode data** — symptoms can be added during the session.
2. **Temporal reasoning inputs** — each symptom has an onset time after exposure.
3. **Rule-based tools** — allergy-vs-flu comparison and emergency-warning checks are deterministic Python functions, not LLM guesses.
4. **Source-linked reference knowledge** — the server can return concise guidance with source URLs.
5. **Medication-allergy logic** — a small demo rule illustrates domain validation.
6. **Generated clinical artifact** — the server produces a structured handoff note.
7. **MCP resources** — the full episode and reference knowledge are addressable as resources.
8. **MCP prompt** — `early_allergy_review` demonstrates a reusable server-side prompt.

The LLM handles orchestration and explanation; the MCP server owns the record, rules, and structured outputs. That separation is the main architectural idea of the example.
